# 02. Preprocessing: Feature Selection and Train/Test Split

## 1. Setup and Data Loading

### 1.1 Importing Libraries

In [1]:
#necessary imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

### 1.2 Loading the Dataset

In [2]:
#loading the dataset
df = pd.read_csv('../data/zenodo_clean.csv')

In [3]:
#shape of the dataset
df.shape

(19251, 91)

## 2. Target Variable

The target is recreated here in the same way as in notebook 01: a student is labelled depressed (1) when the PHQ-9 total is 10 or more, which corresponds to moderate or worse depression. All other students are labelled 0.

### 2.1 Creating the Target Column

In [4]:
#creating the target column
df['depressed'] = (df['phq_total'] >= 10).astype(int)
df['depressed'].value_counts()

depressed
0    17967
1     1284
Name: count, dtype: int64

In [5]:
#percentage of depressed vs not depressed
df['depressed'].value_counts(normalize=True) * 100

depressed
0    93.330217
1     6.669783
Name: proportion, dtype: float64

## 3. Feature Selection

The screening questionnaire has to be short enough to be used in an app, so only some questions can be kept. The choice follows three rules.

1. Leakage. `phq_q3` to `phq_q9`, `phq_total`, `phq_severity` and `safety_flag` are excluded. The target is calculated from the PHQ-9 answers, so these columns define it, and a model that used them would simply read the answer. `phq_q1` and `phq_q2` are kept because they form the PHQ-2, the benchmark rule.
2. Length. The 21 questions with the highest correlation with the PHQ-9 total are chosen, to match the app's goal of about 21 questions.
3. Privacy. Demographic variables (gender, age, education, smoking and drinking) are left out. This keeps the app short and avoids asking students for extra sensitive information. Notebook 01 also showed that gender separates depressed from non-depressed students only weakly.

### 3.1 Candidate Questions

The candidate pool contains the PHQ-2 questions, the 7 GAD-7 questions, the 7 ISI questions and the 14 PSS questions.

In [6]:
#questions we are allowed to use (phq q3-q9 are left out because they make up the answer)
phq_items = ['phq_q1', 'phq_q2']
gad_items = ['gad_q1', 'gad_q2', 'gad_q3', 'gad_q4', 'gad_q5', 'gad_q6', 'gad_q7']
isi_items = ['isi_q1', 'isi_q2', 'isi_q3', 'isi_q4', 'isi_q5', 'isi_q6', 'isi_q7']
pss_items = ['pss_q1', 'pss_q2', 'pss_q3', 'pss_q4', 'pss_q5', 'pss_q6', 'pss_q7', 'pss_q8', 'pss_q9', 'pss_q10', 'pss_q11', 'pss_q12', 'pss_q13', 'pss_q14']
items = phq_items + gad_items + isi_items + pss_items

In [7]:
#number of candidate questions
len(items)

30

#### Observation

The candidate pool has 30 questions: 2 from the PHQ-9, 7 from the GAD-7, 7 from the ISI and 14 from the PSS-14.

### 3.2 Correlation of Each Question with the PHQ-9 Total

Each candidate question is correlated with the PHQ-9 total and the questions are ranked from strongest to weakest. A question that moves closely with the PHQ-9 total carries more information about depression.

In [8]:
#correlation of each question with the phq-9 total
corr = df[items + ['phq_total']].corr()['phq_total'].drop('phq_total').sort_values(ascending=False)
corr

phq_q2     0.723610
phq_q1     0.710337
gad_q6     0.604204
gad_q2     0.589839
gad_q3     0.586533
gad_q4     0.583413
gad_q1     0.581367
pss_q2     0.570011
pss_q3     0.562709
pss_q1     0.534699
gad_q5     0.525208
gad_q7     0.525156
isi_q1     0.493054
isi_q7     0.472410
isi_q2     0.457171
pss_q14    0.454373
isi_q4     0.448606
pss_q11    0.385191
isi_q5     0.383499
isi_q3     0.328578
pss_q8     0.326087
isi_q6     0.318061
pss_q7     0.257100
pss_q13    0.256488
pss_q9     0.232735
pss_q6     0.200826
pss_q5     0.167794
pss_q10    0.155567
pss_q4     0.113506
pss_q12    0.095765
Name: phq_total, dtype: float64

#### Observation

`phq_q2` (0.72) and `phq_q1` (0.71) are the strongest, followed by the GAD-7 questions (0.53 to 0.60), `pss_q2`, `pss_q3` and `pss_q1` (0.53 to 0.57) and the ISI questions (0.32 to 0.49). The weakest questions are PSS items, with `pss_q12` at 0.10. The 21st question, `pss_q8` (0.326), is only just ahead of the 22nd, `isi_q6` (0.318), so the cut-off falls where the ranking is very close and either of the two would be a reasonable choice.

### 3.3 Selecting the Top 21 Questions

The 21 questions with the highest correlation are kept as the model features. Correlations here are calculated on all students before the split, which is a simplification. The closest call in the ranking is the one described above, and the choice is unlikely to change, since the questions at the cut-off have very similar correlations.

In [9]:
#picking the top 21 questions by correlation with phq-9
features = corr.head(21).index.tolist()
features

['phq_q2',
 'phq_q1',
 'gad_q6',
 'gad_q2',
 'gad_q3',
 'gad_q4',
 'gad_q1',
 'pss_q2',
 'pss_q3',
 'pss_q1',
 'gad_q5',
 'gad_q7',
 'isi_q1',
 'isi_q7',
 'isi_q2',
 'pss_q14',
 'isi_q4',
 'pss_q11',
 'isi_q5',
 'isi_q3',
 'pss_q8']

#### Observation

The selected features are the two PHQ-2 questions, seven GAD-7 questions, six ISI questions and six PSS questions. None of them is a PHQ-9 question from 3 to 9, so the leakage rule is respected.

### 3.4 Questions per Questionnaire

The first three letters of each question name (phq, gad, isi or pss) identify its questionnaire. Counting the selected questions by these prefixes shows how long each part of the app would be.

In [10]:
#first three letters of each question name = the questionnaire
prefixes = pd.Series(features).str[:3]

In [11]:
#how many questions from each questionnaire
prefixes.value_counts()

gad    7
pss    6
isi    6
phq    2
Name: count, dtype: int64

#### Observation

All seven GAD-7 questions are selected, together with 6 PSS, 6 ISI and 2 PHQ questions, which makes 21 in total. The anxiety questionnaire is therefore the most completely represented. Only 6 of the 14 PSS questions are kept, since most of the remaining PSS items correlate weakly with the PHQ-9 total.

## 4. Train/Test Split

The students are divided into a training set, used to fit the models, and a test set, used only to measure how well they work on students they have not seen. The split is created once here and saved, so that all models are trained and tested on the same students.

### 4.1 Features and Target

`X` contains the 21 selected questions and `y` is the depression label.

In [12]:
#features (X) and target (y)
X = df[features]
y = df['depressed']

### 4.2 Splitting the Data

Three arguments of `train_test_split` matter here.

- `test_size=0.2` puts 20% of the students in the test set and 80% in the training set.
- `random_state=42` fixes the random shuffle, so the same split is produced every time the notebook is run.
- `stratify=y` keeps the proportion of depressed students the same in both sets. Without it, a random split of such an imbalanced target could leave the test set with noticeably more or fewer depressed students.

In [13]:
#splitting 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(X_train.shape)
print(X_test.shape)

(15400, 21)
(3851, 21)


#### Observation

The training set has 15,400 students and the test set has 3,851, each with 21 features. This is an 80/20 split of the 19,251 students.

### 4.3 Checking the Class Balance

The percentage of depressed students should be almost identical in both sets, and the counts show how many depressed students each set contains.

In [14]:
#checking the split kept the same balance
print(y_train.mean() * 100)
print(y_test.mean() * 100)

6.6688311688311686
6.673591274993508


In [15]:
#number of depressed students in each set
print(y_train.sum())
print(y_test.sum())

1027
257


#### Observation

Stratification worked: 6.67% of the training students and 6.67% of the test students are depressed, matching the 6.67% in the full dataset. The training set contains 1,027 depressed students and the test set contains 257. Because the test set has only 257 depressed students, each one missed or caught moves recall by about 0.4 percentage points, so the test results in notebook 03 carry some uncertainty.

## 5. Note on Scaling

No scaling or standardisation is applied. All features are answers to questionnaire items on small, similar scales, so no feature dominates because of its units. Leaving the answers as they are also keeps the model coefficients easy to read (one coefficient per question, per answer point), and it makes the final model simple to port to Kotlin for the app, since the app can use the raw answers directly.

In [16]:
#smallest and largest answer for each selected question
X_train.describe().loc[['min', 'max']]

,phq_q2,phq_q1,gad_q6,gad_q2,gad_q3,gad_q4,gad_q1,pss_q2,pss_q3,pss_q1,...,gad_q7,isi_q1,isi_q7,isi_q2,pss_q14,isi_q4,pss_q11,isi_q5,isi_q3,pss_q8
min,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,1.0
max,3.0,3.0,3.0,3.0,3.0,3.0,3.0,5.0,5.0,5.0,...,3.0,4.0,4.0,4.0,5.0,4.0,5.0,4.0,4.0,5.0


#### Observation

The PHQ-2 and GAD-7 answers range from 0 to 3, the ISI answers from 0 to 4 and the PSS answers from 1 to 5. All features therefore lie on scales of at most five points, which supports leaving them unscaled.

## 6. Saving the Split

The training and test sets are saved with their labels in the `data` folder. All models (notebooks 03 to 06) load these exact files, so any difference in results comes from the model and not from a different random split. This makes the comparison fair.

In [17]:
#saving the split so every model uses the same data
pd.concat([X_train, y_train], axis=1).to_csv('../data/train.csv', index=False)
pd.concat([X_test, y_test], axis=1).to_csv('../data/test.csv', index=False)

In [18]:
#checking the saved files
print(pd.read_csv('../data/train.csv').shape)
print(pd.read_csv('../data/test.csv').shape)

(15400, 22)
(3851, 22)


#### Observation

`train.csv` has 15,400 rows and 22 columns, and `test.csv` has 3,851 rows and 22 columns. The 22 columns are the 21 questions plus the `depressed` label. The row counts match the split in Section 4.2, so the files were written correctly.

The next notebook, `03_logistic_regression.ipynb`, loads these two files and fits the first model.